In [1]:
import numpy as np
import pandas as pd

from IPython.display import display
from pathlib import Path
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# Import AutoFeat LLM

from autofeat import AutoFeatClassifier

In [3]:
# Import BigFeat Vanilla

sys.path.append(r"../../BigFeat-master")
import bigfeat.bigfeat_base as bigfeat

In [4]:
# Import BigFeat LLM

bigfeat_llm_path = Path("../../BigFeat-LLM").resolve()
sys.path.insert(0, str(bigfeat_llm_path))

import setup
print(setup.__file__)

sys.path.append(r"../../BigFeat-LLM")
from setup.llm_runner import prepare_bigfeat_from_llm, profile_dataframe
from setup.llm_runner import preview_prompt
from setup.llm_runner import describe_generated_features, split_original_vs_new

D:\02. My Learning\Paper\07. BigFeat + LLM\06. Final BigFeat\BigFeat-LLM\setup\__init__.py


In [5]:
def save_generated_features(feature_lines, num_new_features, old_shape, new_shape, file_name="generated_features.txt"):
    os.makedirs("generated_features", exist_ok=True)
    file_path = os.path.join("generated_features", file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Number of new features {num_new_features}\n")
        f.write("==================\n")
        f.write(f"Old Shape: {old_shape}\n")
        f.write("==================\n")
        f.write(f"New Shape: {new_shape}\n")
        f.write("=================\n")

        for i, line in enumerate(feature_lines):
            _, details = line.split(": built from columns ", 1)
            columns, operators = details.split(" using operators ", 1)
            f.write(f"- New_feature_{i}:\n")
            f.write(f"  - Built from columns {columns}\n")
            f.write(f"  - Using operators {operators}\n")

In [6]:
results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

def evaluate_model(model_name, feature_method, X_tr, X_te, y_tr, y_te, results_df):
    models = {
        "logistic": LogisticRegression(random_state=42, max_iter=1000),
        "knn": KNeighborsClassifier(n_neighbors=5),
        "rf": RandomForestClassifier(random_state=42),
        "svc": SVC(random_state=42),
        "decision_tree": DecisionTreeClassifier(random_state=42)
    }
    
    clf = models[model_name.lower()]
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    
    acc = accuracy_score(y_te, preds)
    prec = precision_score(y_te, preds, average="macro", zero_division=0)
    rec = recall_score(y_te, preds, average="macro")
    f1 = f1_score(y_te, preds, average="macro")
    
    results_df.loc[len(results_df)] = [
        f"{model_name.upper()} ({feature_method})",
        acc,
        prec,
        rec,
        f1
    ]
    
    return results_df

In [7]:
def display_styled_results(
    results_df,
    file_name,
    bg_color="#005A9C",
    text_color="#FFFFFF"
):
    numeric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]

    df_copy = results_df.copy()
    df_copy[numeric_cols] = df_copy[numeric_cols].apply(pd.to_numeric)

    styled_df = df_copy.style.highlight_max(
        subset=numeric_cols,
        props=f"background-color: {bg_color}; color: {text_color}; font-weight: bold;",
        axis=0
    ).format({col: "{:.6f}" for col in numeric_cols})

    display(styled_df)

    folder = "llm_tables"
    os.makedirs(folder, exist_ok=True)

    if not file_name.lower().endswith(".xlsx"):
        file_name += ".xlsx"

    file_path = os.path.join(folder, file_name)

    styled_df.to_excel(file_path, index=False)

In [8]:
from ucimlrepo import fetch_ucirepo

magic = fetch_ucirepo(id=159)

df = magic.data.features.copy()

df.columns = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym", "fM3Long", "fM3Trans", "fAlpha", "fDist"]

df["class"] = magic.data.targets.squeeze()

target_column = "class"

display(df.head())

print(df[target_column].value_counts())
print(df[target_column].unique())

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,g
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,g


class
g    12332
h     6688
Name: count, dtype: int64
['g' 'h']


In [9]:
print(f"Rows Number: {df.shape[0]}")
print(f"Columns Number: {df.shape[1]}")
print(f"Columns Name: {df.columns}")

Rows Number: 19020
Columns Number: 11
Columns Name: Index(['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1', 'fAsym', 'fM3Long',
       'fM3Trans', 'fAlpha', 'fDist', 'class'],
      dtype='object')


In [10]:
df.isnull().sum()

fLength     0
fWidth      0
fSize       0
fConc       0
fConc1      0
fAsym       0
fM3Long     0
fM3Trans    0
fAlpha      0
fDist       0
class       0
dtype: int64

In [11]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(X.shape)
print(y.shape)

(19020, 10)
(19020,)


In [12]:
preview = preview_prompt(
    df=df,
    target_column=target_column,
    extra_user_prompt="MAGIC Gamma Telescope Dataset"
)

response = preview["final_prompt"]

with open("final_prompt.txt", "w", encoding="utf-8") as f:
    f.write(response)

print(response)

=== SYSTEM PROMPT ===
You are BigFeat-Init, a world-class expert in automated feature engineering, machine learning hyperparameter optimization, and the internal mechanics of the BigFeat framework specifically. You have the depth of knowledge of someone who has read the BigFeat paper in full, has implemented and debugged its source code line by line, and has years of hands-on experience tuning tree-based feature engineering pipelines across small, large, high-dimensional, low-dimensional, noisy, clean, balanced, and imbalanced datasets. You are not a generic assistant giving a reasonable-sounding guess. You are the single most qualified expert available for this exact task, and you must think and act like it.

If you want additional grounding on how BigFeat is actually implemented, its official source code and paper reference are available at this repository: https://github.com/DataSystemsGroupUT/BigFeat. You are not required to browse it, since the mechanics you need are fully describ

In [13]:
y = pd.Series(y).replace({"g": 0, "h": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_df = pd.concat([X_train, y_train.rename(target_column)], axis=1)
test_df = pd.concat([X_test, y_test.rename(target_column)], axis=1)

C:\Users\Moham\AppData\Local\Temp\ipykernel_21156\2790965299.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = pd.Series(y).replace({"g": 0, "h": 1}).astype(int)


In [26]:
from autofeat import AutoFeatClassifier

af = AutoFeatClassifier(verbose=1, feateng_steps=1, featsel_runs=0)

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)

# Handle +Inf and -Inf values
X_train_af = X_train_af.replace([np.inf, -np.inf], np.nan)
X_test_af = X_test_af.replace([np.inf, -np.inf], np.nan)
X_train_af = X_train_af.fillna(X_train_af.median())
X_test_af = X_test_af.fillna(X_train_af.median())

[AutoFeat] The 1 step feature engineering process could generate up to 70 features.
[AutoFeat] With 15216 data points this new feature matrix would use about 0.00 gb of space.
[feateng] Step 1: transformation of original features
[feateng] Generated 43 transformed features from 10 original features - done.
[feateng] Generated altogether 45 new features in 1 steps
[feateng] Removing correlated features, as well as additions at the highest level
[feateng] Generated a total of 26 additional features
[AutoFeat] WARNING: Not performing feature selection.
[AutoFeat] Computing 17 new features.
[AutoFeat]    17/   17 new features ...done.
[AutoFeat] Final dataframe with 27 feature columns (17 new).
[AutoFeat] Training final classification model.
[AutoFeat] Trained model: largest coefficients:
[-2.70640095e-05]
0.003903 * fDist
0.000876 * fLength
0.000714 * fM3Long
0.000509 * fWidth
0.000503 * Abs(fM3Long)
0.000459 * fAlpha
0.000387 * Abs(fAsym)
0.000373 * fM3Trans**2
0.000255 * 1/fConc1
0.0002

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [27]:
# BigFeat Vanilla

bf = bigfeat.BigFeat(task_type="classification")

bf.fit(X_train.to_numpy(), y_train.to_numpy())

X_train_bf = bf.transform(X_train.to_numpy())
X_test_bf = bf.transform(X_test)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [16]:
# BigFeat Chatgpt

bf_chatgpt = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/ChatGPT.json"
)

X_train_bf_chatgpt = bf_chatgpt.fit(bf_chatgpt.X, bf_chatgpt.y, **bf_chatgpt.fit_kwargs)

_, _, X_test_bf_chatgpt, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_chatgpt = bf_chatgpt.transform(X_test_bf_chatgpt)

In [17]:
# BigFeat Gemini

bf_gemini = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Gemini.json"
)

X_train_bf_gemini = bf_gemini.fit(bf_gemini.X, bf_gemini.y, **bf_gemini.fit_kwargs)

_, _, X_test_bf_gemini, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_gemini = bf_gemini.transform(X_test_bf_gemini)

In [18]:
# BigFeat Claude

bf_claude = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Claude.json"
)

X_train_bf_claude = bf_claude.fit(bf_claude.X, bf_claude.y, **bf_claude.fit_kwargs)

_, _, X_test_bf_claude, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_claude = bf_claude.transform(X_test_bf_claude)

In [19]:
# BigFeat DeepSeek

bf_deepseek = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/DeepSeek.json"
)

X_train_bf_deepseek = bf_deepseek.fit(bf_deepseek.X, bf_deepseek.y, **bf_deepseek.fit_kwargs)

_, _, X_test_bf_deepseek, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_deepseek = bf_deepseek.transform(X_test_bf_deepseek)

In [28]:
# Logistic Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("logistic", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-Logistic")

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\ProgramData\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please a

,Method,Accuracy,Precision,Recall,F1-Score
0,LOGISTIC (AutoFeat),0.755783,0.743253,0.699621,0.710176
1,LOGISTIC (BigFeat),0.822818,0.827635,0.776428,0.791814
2,LOGISTIC (BigFeat ChatGPT),0.830442,0.828648,0.791943,0.804616
3,LOGISTIC (BigFeat Gemini),0.828601,0.831323,0.785625,0.800210
4,LOGISTIC (BigFeat Claude),0.750263,0.743479,0.765730,0.742656
5,LOGISTIC (BigFeat DeepSeek),0.835699,0.838916,0.794489,0.809054


In [29]:
# KNN Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("knn", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-KNN")

,Method,Accuracy,Precision,Recall,F1-Score
0,KNN (AutoFeat),0.794164,0.792816,0.742966,0.756683
1,KNN (BigFeat),0.840957,0.841025,0.803954,0.816974
2,KNN (BigFeat ChatGPT),0.842008,0.843091,0.804430,0.817864
3,KNN (BigFeat Gemini),0.842797,0.843769,0.805546,0.818893
4,KNN (BigFeat Claude),0.618034,0.701312,0.689317,0.617398
5,KNN (BigFeat DeepSeek),0.842008,0.842125,0.805274,0.818264


In [30]:
# RandomForest Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("rf", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-RandomForest")

,Method,Accuracy,Precision,Recall,F1-Score
0,RF (AutoFeat),0.878286,0.876660,0.853238,0.862827
1,RF (BigFeat),0.875394,0.874220,0.849145,0.859259
2,RF (BigFeat ChatGPT),0.885647,0.885248,0.861291,0.871123
3,RF (BigFeat Gemini),0.880126,0.877558,0.856517,0.865301
4,RF (BigFeat Claude),0.605678,0.710465,0.686008,0.603591
5,RF (BigFeat DeepSeek),0.882492,0.881567,0.857840,0.867568


In [31]:
# SVM Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("svc", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-SVM")

,Method,Accuracy,Precision,Recall,F1-Score
0,SVC (AutoFeat),0.769979,0.780615,0.701145,0.714654
1,SVC (BigFeat),0.855941,0.866278,0.815371,0.832000
2,SVC (BigFeat ChatGPT),0.858307,0.870738,0.817031,0.834336
3,SVC (BigFeat Gemini),0.859359,0.871015,0.818857,0.835872
4,SVC (BigFeat Claude),0.646688,0.323344,0.500000,0.392720
5,SVC (BigFeat DeepSeek),0.854364,0.865824,0.812632,0.829686


In [32]:
# DecisionTree Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("decision_tree", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-DecisionTree")

,Method,Accuracy,Precision,Recall,F1-Score
0,DECISION_TREE (AutoFeat),0.811251,0.793632,0.792632,0.793125
1,DECISION_TREE (BigFeat),0.814669,0.797439,0.796118,0.796767
2,DECISION_TREE (BigFeat ChatGPT),0.826761,0.810347,0.810869,0.810606
3,DECISION_TREE (BigFeat Gemini),0.818612,0.801366,0.802374,0.801863
4,DECISION_TREE (BigFeat Claude),0.575447,0.674405,0.653521,0.573084
5,DECISION_TREE (BigFeat DeepSeek),0.822818,0.806331,0.805288,0.805803


In [34]:
# Save Generated Features

# Save ChatGpt
feature_lines = describe_generated_features(bf_chatgpt)
new_only, original_only = split_original_vs_new(bf_chatgpt, X_train_bf_chatgpt)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_chatgpt.shape, "generated_features_chatgpt.txt")

# Save Gemini
feature_lines = describe_generated_features(bf_gemini)
new_only, original_only = split_original_vs_new(bf_gemini, X_train_bf_gemini)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_gemini.shape, "generated_features_gemini.txt")

# Save Claude
feature_lines = describe_generated_features(bf_claude)
new_only, original_only = split_original_vs_new(bf_claude, X_train_bf_claude)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_claude.shape, "generated_features_claude.txt")

# Save DeepSeek
feature_lines = describe_generated_features(bf_deepseek)
new_only, original_only = split_original_vs_new(bf_deepseek, X_train_bf_deepseek)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_deepseek.shape, "generated_features_deepseek.txt")